# 03. Tạo bộ dữ liệu cho Detection (YOLO) và OCR

Đầu ra:
```
<datasets_dir>/vnlp_det/  images/{train,val,test}  labels/{train,val,test}  data.yaml
                          2 lớp: 0 = plate_1row, 1 = plate_2row
<datasets_dir>/vnlp_ocr/  {train,val,test}/*.jpg   {train,val,test}.csv  (filename, label, ...)
```
`datasets_dir` mặc định là `E:/vietnamese-license-plate-recognition/datasets`: dữ liệu sinh ra nằm cùng ổ với VNLP,
không chép sang ổ D chứa project.
- Ảnh detection dùng **hard link** (không tốn thêm dung lượng, cần cùng ổ với VNLP), lỗi thì chép.
- Ảnh OCR là vùng biển cắt theo hộp nhãn, có thêm lề `PAD`.
- Ảnh bị lỗi trong `reports/image_check.csv` (notebook 02) sẽ bị bỏ qua.

In [ ]:
import sys, csv, json, hashlib, collections, os, shutil, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from alpr.paths import load_paths

PATHS = load_paths()                  # khai báo trong config.json (xem config.example.json)
VNLP = PATHS["vnlp_root"]             # dữ liệu gốc (ổ E:)
DATASETS = PATHS["datasets_dir"]      # dataset sinh ra (để cùng ổ với VNLP cho hard link)
EXTERNAL = PATHS["external_dir"]      # video, bộ dữ liệu ngoài
manifest = pd.read_csv(ROOT / "data/labels/vnlp_manifest.csv", dtype={"label": str, "province": str}).fillna("")
print(len(manifest), "dòng manifest")

In [ ]:
OUT = DATASETS
MODE = "link"        # "link" hoặc "copy"
PAD = 0.05           # lề quanh biển khi cắt ảnh OCR (5%)
CLASSES = {"1row": 0, "2row": 1}

skip = set()
if (ROOT / "reports/image_check.csv").exists():
    ic = pd.read_csv(ROOT / "reports/image_check.csv").fillna("")
    skip = set(ic.rel_path[ic.issue != ""])
print("Ảnh bị bỏ qua do lỗi:", len(skip))

def place(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    if MODE == "link":
        try:
            os.link(src, dst); return
        except OSError:
            pass
    shutil.copy2(src, dst)

## 1. Sinh dữ liệu

In [ ]:
det, ocr = OUT / "vnlp_det", OUT / "vnlp_ocr"
ocr_rows = collections.defaultdict(list)
n_det = n_ocr = n_skip = 0
for i, r in enumerate(manifest.itertuples()):
    src = VNLP / r.rel_path.lstrip("/")
    if r.rel_path in skip or not src.exists():
        n_skip += 1; continue
    name = Path(r.rel_path).name
    with Image.open(src) as im:
        W, H = im.size
        # detection
        place(src, det / "images" / r.split / name)
        cx, cy, bw, bh = (r.x1 + r.x2) / 2 / W, (r.y1 + r.y2) / 2 / H, (r.x2 - r.x1) / W, (r.y2 - r.y1) / H
        lbl = det / "labels" / r.split / (Path(name).stem + ".txt")
        lbl.parent.mkdir(parents=True, exist_ok=True)
        lbl.write_text(f"{CLASSES[r.plate_type]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
        n_det += 1
        # OCR
        if r.use_for_ocr == 1:
            px, py = (r.x2 - r.x1) * PAD, (r.y2 - r.y1) * PAD
            box = (max(0, int(r.x1 - px)), max(0, int(r.y1 - py)), min(W, int(r.x2 + px)), min(H, int(r.y2 + py)))
            dst = ocr / r.split / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                im.convert("RGB").crop(box).save(dst, quality=95)
            ocr_rows[r.split].append(dict(filename=name, label=r.label, plate_type=r.plate_type,
                                          vehicle=r.vehicle, province=r.province))
            n_ocr += 1
    if (i + 1) % 5000 == 0:
        print(f"  {i + 1}/{len(manifest)}")

(det / "data.yaml").write_text(
    f"path: {det.resolve().as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
    "names:\n  0: plate_1row\n  1: plate_2row\n", encoding="utf-8")
for split, rows in ocr_rows.items():
    pd.DataFrame(rows).to_csv(ocr / f"{split}.csv", index=False, encoding="utf-8")
print(f"Detection: {n_det} ảnh | OCR: {n_ocr} ảnh biển | Bỏ qua: {n_skip}")

## 2. Kiểm tra lại bằng mắt

Đọc ngược tệp nhãn YOLO vừa sinh và vẽ lên ảnh. Hộp phải ôm sát biển.

In [ ]:
def show_yolo(split="train", n=12, seed=0):
    imgs = sorted((det / "images" / split).glob("*.jpg"))
    random.Random(seed).shuffle(imgs)
    fig, axes = plt.subplots(2, n // 2, figsize=(3 * n // 2, 5))
    for ax, p in zip(axes.ravel(), imgs[:n]):
        im = Image.open(p); W, H = im.size
        c, cx, cy, bw, bh = map(float, (det / "labels" / split / (p.stem + ".txt")).read_text().split())
        ax.imshow(im); ax.axis("off")
        ax.add_patch(plt.Rectangle(((cx - bw / 2) * W, (cy - bh / 2) * H), bw * W, bh * H,
                                   fill=False, ec="red" if c == 0 else "lime", lw=2))
        ax.set_title(["1 dòng", "2 dòng"][int(c)], fontsize=9)
    plt.tight_layout(); plt.show()

show_yolo("train")

In [ ]:
def show_ocr(split="train", n=24, seed=0):
    df = pd.read_csv(ocr / f"{split}.csv", dtype=str).sample(n, random_state=seed)
    fig, axes = plt.subplots(4, n // 4, figsize=(2.4 * n // 4, 6))
    for ax, r in zip(axes.ravel(), df.itertuples()):
        ax.imshow(Image.open(ocr / split / r.filename)); ax.axis("off"); ax.set_title(r.label, fontsize=9)
    plt.tight_layout(); plt.show()

show_ocr("train")

In [ ]:
# Thống kê nhanh bộ OCR
for split in ["train", "val", "test"]:
    d = pd.read_csv(ocr / f"{split}.csv", dtype=str)
    print(f"{split:5s}: {len(d):6d} ảnh | 1 dòng {(d.plate_type == '1row').sum():6d} | "
          f"2 dòng {(d.plate_type == '2row').sum():6d} | ngoài mã 36: {(d.province != '36').sum()}")
chars = sorted(set("".join(pd.read_csv(ocr / "train.csv", dtype=str).label)))
print("Bộ ký tự:", "".join(chars), f"({len(chars)} ký tự)")